# 03 · The full voice loop (this is "live")

Two loops run **at the same time** on one socket:

```
mic ──100 ms chunks──► [send_loop] ──input_audio_buffer.append──► OpenAI
                                                                    │  semantic VAD decides
speaker ◄──buffer◄──── [recv_loop] ◄──response.output_audio.delta───┘  when you stopped
```
No "send" button: the **server** detects the end of your turn (VAD) and starts answering by itself.

🎧 **Wear headphones.** Otherwise the model hears itself through the mic.

New helpers: `audio.mic_chunks`, `audio.file_chunks`, `audio.Speaker`, `realtime.start_session`, `realtime.converse`.

In [1]:
import asyncio, inspect
import numpy as np
from IPython.display import Audio
from live import audio, realtime

## 1. The mic as an async stream
PortAudio calls us from **its own thread** every 100 ms. `call_soon_threadsafe` hands each chunk to the asyncio loop through a queue. Let's grab 3 chunks:

In [2]:
mic = audio.mic_chunks()
first = [await anext(mic) for _ in range(3)]
await mic.aclose()
[len(c) for c in first], audio.rms(audio.from_bytes(first[0]))

([4800, 4800, 4800], 0.000697135750669986)

2400 samples × 2 bytes = 4800 bytes = 100 ms. Same numbers as notebook 01, but now they arrive continuously.

## 2. The speaker as a buffer
Model audio arrives **faster than real time** (notebook 02: 6.75 s of speech in 1.6 s), so it piles up in a buffer that the sound card drains at 24 kHz. Watch it drain:

In [3]:
spk = audio.Speaker(); spk.stream.start()
beep = audio.float_to_pcm16(0.2 * np.sin(2 * np.pi * 440 * np.arange(audio.RATE) / audio.RATE))  # 1 s, 440 Hz
spk.write(audio.to_bytes(beep))
before = spk.buffered_seconds()
await asyncio.sleep(0.5)
before, round(spk.buffered_seconds(), 2)

(1.0, 0.53)

That leftover buffer is why barge-in needs `spk.clear()`. Otherwise the model keeps talking after you interrupt (notebook 04).

In [4]:
await asyncio.sleep(0.6); spk.buffered_seconds()

0.0

## 3. The loop itself
Two coroutines plus a timeout. This is the whole "live" engine:

In [5]:
print(inspect.getsource(realtime.converse))

async def converse(ws, chunks, speaker, seconds: float) -> list[tuple]:
    """Full duplex for `seconds`: stream `chunks` (async iterator of PCM16 bytes) up while
    playing model audio down. Returns a log of (t, event, detail); also prints it live."""
    t0, log, responding = time.perf_counter(), [], False

    def note(event, detail=""):
        log.append((round(time.perf_counter() - t0, 2), event, detail))
        print(*log[-1])

    async def send_loop():  # mic → server, forever
        async for c in chunks:
            await send(ws, {"type": "input_audio_buffer.append", "audio": audio.to_b64(c)})

    async def recv_loop():  # server → speaker, forever
        nonlocal responding
        async for raw in ws:
            e = json.loads(raw)
            if e["type"] == "error":
                raise RuntimeError(e["error"])
            if e["type"] == "response.output_audio.delta":
                if not responding:
                    responding = True
                    no

## 4. Simulated user (verifiable without talking)
The server can't tell a mic from any other source of chunks. So: synthesize a spoken question with Windows' built-in TTS (`audio.say`) and feed it through `file_chunks`, exactly as if it came from the mic.

(We first tried making the Realtime model read the question aloud. It sometimes *answered* it instead. A plain TTS engine is deterministic.)

In [6]:
question = audio.say("What is the tallest mountain in the world?")
question.shape, question.dtype, round(len(question) / audio.RATE, 2)

((69468,), dtype('int16'), 2.89)

In [7]:
Audio(question, rate=audio.RATE)

## 5. Turn detection: who decides you're done?
Two server-side modes:

In [8]:
realtime.SERVER_VAD, realtime.SEMANTIC_VAD

({'type': 'server_vad', 'silence_duration_ms': 500}, {'type': 'semantic_vad'})

We use `server_vad` (500 ms of silence ends the turn) because it is predictable. A fresh session:

In [9]:
ws = await realtime.start_session("You are a friendly voice tutor. Answer in one short sentence.")
ws.state

<State.OPEN: 1>

Run the loop for (question length + 10 s). The fake mic sends the question and then silence. Watch the server detect speech start and stop, commit the audio, and answer **on its own**:

In [10]:
sim_log = await realtime.converse(ws, audio.file_chunks(question), spk, barge_in=False, seconds=len(question) / audio.RATE + 10)

0.44 input_audio_buffer.speech_started 


3.28 input_audio_buffer.speech_stopped 
3.28 input_audio_buffer.committed 
3.29 response.created in_progress


3.73 first_audio_delta 


4.4 model_said The tallest mountain in the world is Mount Everest.
4.4 response.done completed


In [11]:
sim_log  # (seconds, event, detail)

[(0.44, 'input_audio_buffer.speech_started', ''),
 (3.28, 'input_audio_buffer.speech_stopped', ''),
 (3.28, 'input_audio_buffer.committed', ''),
 (3.29, 'response.created', 'in_progress'),
 (3.73, 'first_audio_delta', ''),
 (4.4, 'model_said', 'The tallest mountain in the world is Mount Everest.'),
 (4.4, 'response.done', 'completed')]

## 6. Where the felt latency goes
① VAD wait = from your real last word to `speech_stopped`. ② Model = from `speech_stopped` to the first audio delta.

In [12]:
t = {ev: ts for ts, ev, _ in sim_log}
real_speech_end = len(question) / audio.RATE      # the fake mic finished the question here
vad_wait = round(t['input_audio_buffer.speech_stopped'] - real_speech_end, 2)
model_latency = round(t['first_audio_delta'] - t['input_audio_buffer.speech_stopped'], 2)
felt_latency = round(vad_wait + model_latency, 2)
vad_wait, model_latency, felt_latency

(0.39, 0.45, 0.84)

For comparison, a first run with `semantic_vad` had a VAD wait of about 4.8 s on this same kind of question. That is why the default here is `server_vad`. Lower `silence_duration_ms` means snappier replies but more cut-offs mid-sentence.

## 7. Your turn: live mic 🎤
Run the cell below and talk. Ask something, pause, and listen; repeat. It stops after `LIVE_SECONDS`.
(Don't talk over the model yet: the interruption fix is notebook 04.)

In [13]:
LIVE_SECONDS = 30

In [14]:
live_log = await realtime.converse(ws, audio.mic_chunks(), spk, barge_in=False, seconds=LIVE_SECONDS)

In [15]:
live_log

[]

In [16]:
await ws.close(); spk.stream.close()
ws.state

<State.CLOSED: 3>

## Takeaways
- Live = **two concurrent loops** + **server-side VAD**. No button, no "end of message".
- The mic and the speaker run on audio threads. Queues and buffers bridge them to asyncio.
- Felt latency = VAD end-of-speech wait + time to first audio delta.
- Known gap: if you talk over the model, already-buffered audio keeps playing.

**Next (04):** barge-in (clear the speaker + `conversation.item.truncate`) and live transcripts of both sides.